# Custom sequence constraints

[Notebook](custom-constraints.ipynb)

Pass a function to `constraint=` that receives a batch of sequences and returns one Boolean per sequence. `True` allows the sequence. The constraint applies to seeds and proposed candidates. To exclude a pattern on both DNA strands, include its reverse complement.

## Define the filters



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import re

from MotifSift import generate_designs, random_seed_sequences

forbidden = ("GGTACC", "GGATCC")
homopolymer = re.compile(r"([ACGT])\1{5}")

def gc_fraction(sequence):
    return (sequence.count("G") + sequence.count("C")) / len(sequence)

def constraint(sequences):
    return [
        0.35 <= gc_fraction(sequence) <= 0.65
        and not any(pattern in sequence for pattern in forbidden)
        and homopolymer.search(sequence) is None
        for sequence in sequences
    ]

constraint(["AGTACGATACGT", "AAGGATCCTTAC", "AAAATATATTAT", "GGGGGGATATAT"])


[True, False, False, False]



## Optimize with the constraint



In [2]:
class GCountScorer:
    def score(self, sequences):
        return [sequence.count("G") for sequence in sequences]

    def score_soft(self, probabilities):
        return probabilities[:, :, 2].sum(dim=1)

length = 24
seeds = random_seed_sequences(length, 8)
designs = generate_designs(
    GCountScorer(), length=length, initial=seeds, objective="maximize",
    num_candidates=8, score_threshold=10, constraint=constraint,
)
designs.assign(gc_fraction=designs.sequence.map(gc_fraction))[
    ["sequence", "algorithm", "score", "gc_fraction"]
].head()


,sequence,algorithm,score,gc_fraction
0,TGGCCAAAATGTGGTGGGGTCTGA,adalead,10.0,0.541667
1,CGTCCTTTCGTGTGGCTAGGTGCC,adalead,8.0,0.625000
2,ATGCCTAGAAGTGTGTGATCGCAT,adalead,7.0,0.458333
3,GCTATAGGTCTGTATTACGAGGTT,adalead,7.0,0.416667
4,CTGATGTAATAGACCCCAAAAGGG,adalead,6.0,0.458333




## Keep selected positions fixed

`fixed_region` uses zero-based positions and Python slices. In `slice(0, 4)`, the index 4 is excluded. Every starting sequence must already agree at the fixed positions. This example keeps the first four bases and the last three bases from a shared template.



In [3]:
prefix, suffix = "ACGT", "TAC"
initial = [prefix + seed[len(prefix):-len(suffix)] + suffix for seed in seeds]
fixed_designs = generate_designs(
    GCountScorer(), length=length, initial=initial, objective="maximize",
    num_candidates=8, score_threshold=10,
    fixed_region=[slice(0, len(prefix)), slice(-len(suffix), None)],
)
fixed_designs[["algorithm", "sequence", "score"]].head()


,algorithm,sequence,score
0,adalead,ACGTGAGGGGTGTATGATGTGTAC,10.0
1,adalead,ACGTTAGGGCTGCTTGAGGTGTAC,9.0
2,adalead,ACGTCAAAATGTGGTGGGGTCTAC,8.0
3,adalead,ACGTATGCGGCCGGGCTCCTCTAC,7.0
4,adalead,ACGTCTCTCGTGTGGCTAGGTTAC,7.0
